# Step 2 青空文庫からの再構築(1) — 書誌の典拠と XML マークアップ

> **用語の予習・復習**: `docs/glossary.md` の Step 2 を参照。キーワードを見て自分で説明してみてから読むこと。

## このステップの到達目標

1. 青空文庫の書誌索引から，**典拠のある**メタデータを自動生成できる
2. プレーンテクスト版と XHTML 版の違いを説明できる
3. 外字（JIS X 0213）を面区点から Unicode に復元する仕組みを理解する
4. ルビ・注記・会話を XML でタグ付けした版を作る

## 導入：なぜ「削る」のをやめるのか

Step 1 で見たとおり，v1 の欠陥はすべて**削除**に由来する。

```
※［＃「广＋(炎/鳥)」、第4水準2-81-40］     ← 青空文庫の原文
※                                        ← ［＃…］ だけ削った結果（文字が消える）
㽷                                        ← 正しくは面区点から復元できる
```

削除は不可逆である。いったん削ったものは戻らない。したがって

> **本文とメタ情報を「分離」し，何を数えるかは後段の設定で決める**

という設計に変える。これは TEI（Text Encoding Initiative）の基本思想でもある。

## 青空文庫の2つの配布形式

| | プレーンテクスト版 (.txt) | XHTML 版 (.html) |
|---|---|---|
| 文字コード | Shift_JIS | UTF-8 |
| ルビ | `渋江《しぶえ》` | `<ruby><rb>渋江</rb><rt>しぶえ</rt></ruby>` |
| 注記 | `［＃…］` | `<span class="notes">［＃…］</span>` |
| 外字 | `※［＃…］` | `<img class="gaiji" alt="※(噓, 1-84-7)">` |
| 奥付 | 本文と同じ平文 | `<div class="bibliographical_information">` |

**XHTML を使う。** 境界が要素で決まるので，正規表現の当て推量が要らない。


In [ ]:
# ---- 共通の準備（毎回このセルから実行する）----------------------------
import os, sys, csv, json, math, random, shutil, subprocess, warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore', category=FutureWarning)

# リポジトリのルートを自動で探す（notebooks/ から1つ上）
ROOT = Path.cwd()
while not (ROOT / 'config' / 'pipeline.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)

# 日本語フォント（□ にならないように）
for cand in ['Hiragino Sans', 'Yu Gothic', 'Meiryo',
             'Noto Sans CJK JP', 'IPAexGothic', 'MS Gothic']:
    if cand in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = cand
        break
else:
    print('[!] 日本語フォントが見つかりません。docs/00_setup_students.md §1.7 を参照。')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ---- 図はすべて SVG（ベクタ）で保存する ---------------------------------
# 論文・スライドに載せる図は拡大しても劣化してはならない。PNG は解像度が
# 固定されるので，投影や印刷で文字が潰れる。SVG なら任意の倍率で鮮明で，
# Illustrator / Inkscape で軸ラベルだけを直すこともできる。
FIG_EXT   = 'svg'
RASTER_DPI = 200          # rasterized=True の要素にだけ効く
plt.rcParams['svg.fonttype']       = 'path'   # 文字をアウトライン化して環境非依存に
plt.rcParams['savefig.transparent'] = False
# 画面へのインライン表示は既定（PNG）のままにする。
# InlineBackend.figure_formats を 'svg' に変えると，JupyterLab や
# VS Code の版によっては図がまったく表示されなくなることがある。
# **保存されるファイルは SVG** なので，論文・スライドに使うほうは
# ベクタで手元に残る。画面で拡大して見たいときは save_fig が表示する
# パスの .svg をブラウザで開くこと。

def need(path, hint=''):
    """必要な入力があるか確かめる。無ければ**理由を表示して** False を返す。

    セルを `if p.exists():` で囲むと，入力が無いときに何も起きない。
    学生には「壊れている」と「まだ前の工程を走らせていない」の区別が
    つかず，図が出ないという相談の大半がこれである。必ず理由を出す。
    """
    p = Path(path)
    try:
        ok = p.is_file() or (p.is_dir() and any(p.iterdir()))
    except OSError:
        ok = False
    if not ok:
        print(f'[未実行] {p} がありません。')
        if hint:
            print(f'         {hint}')
        print('         この Step の前のセルを上から順に実行すること。'
              '\n         それでも出ない場合は，前の Step のノートブックが'
              '最後まで通っているか確認する。')
    return ok


def load_meta(path=None, analysis_only=True):
    """メタデータを読む。既定では**分析に使う行だけ**を返す。

    落とすのは2種類。書誌としては残すが，集計に足してはいけない行である。
      superseded … v1 の合本。増補で分冊ごとに取り直したので，足すと
                   同じ作品を二重に数える
      merged     … 分冊。03b で canonical の巻に本文を統合したので，
                   この行はもう本文を持たない（『夜明け前』『家』）
      too_short  … 1チャンクにも満たず，チャンク単位の分析に乗らない

    生の表がほしいときは ``analysis_only=False``。
    """
    df = pd.read_csv(path or META)
    if analysis_only and 'completeness' in df.columns:
        drop = df['completeness'].isin(['superseded', 'merged', 'too_short'])
        if drop.any():
            names = '，'.join(df.loc[drop, 'title_aozora'].astype(str))
            print(f'[meta] 分析から除外 {int(drop.sum())} 行: {names}')
        df = df[~drop].reset_index(drop=True)
    return df


def w_ljust(text, width):
    """全角を2桁と数えて左詰めする。

    ``f'{s:<26}'`` は**文字数**で詰めるので，日本語の作品名を並べると
    桁が揃わない（全角は2桁ぶんの幅を占める）。表として読ませるなら
    表示幅で詰めること。

    **なお，一覧を出すなら ``show()`` で表にするほうがよい**（下記）。
    この関数は，表にしにくいもの（KWIC の前後文脈など）を print で
    並べるときに使う。
    """
    import unicodedata
    text = str(text)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    return text + ' ' * max(0, width - w)


# ---------------------------------------------------------------------------
# 分析結果の表示
# ---------------------------------------------------------------------------
# **一覧は print ではなく表で出す。**
#   * print は桁が揃わない（全角の幅）。数字の比較がしにくい
#   * 列に名前が付かないので，あとで見返したときに何の数字か分からない
#   * 並べ替えも絞り込みもできない
# 表にすると，列名がそのまま「何を測ったか」の記録になる。
# **ただし何でも表にするのではない。** 単発の数値・警告・KWIC の前後文脈は
# 文のほうが読みやすい。目安は「2列以上あるか」「行が並ぶか」。
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('caption-side', 'top'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '0 0 .4em 0'),
               ('color', '#33322e'), ('font-size', '95%')]},
    {'selector': 'th',
     'props': [('background', '#f2f2ef'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '.26em .7em'),
               ('border-bottom', '1px solid #c6c5bd'), ('white-space', 'nowrap')]},
    {'selector': 'td',
     'props': [('padding', '.22em .7em'), ('border-bottom', '1px solid #ecebe6')]},
    {'selector': 'tbody tr:hover td', 'props': [('background', '#f7f7f4')]},
]


def show(df, caption='', fmt=None, index=False, header=True, na='—', align=None):
    """DataFrame を表として表示する。Jupyter 以外でも落ちない。

    ``fmt`` は pandas の ``Styler.format`` に渡す辞書
    （例 ``{'一致率': '{:.1%}', 'G²': '{:.0f}'}``）。
    数値の列は自動で右寄せにする。``align`` で列ごとに寄せを指定できる。
    KWIC の左文脈を ``align={'左文脈': 'right'}`` にすると，
    **キーワードが縦に揃う**（等幅フォントに頼らずに揃う）。
    返り値は ``df`` なので ``t = show(df)`` として続けて使える。
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    try:
        from IPython.display import display as _display
        st = df.style.format(fmt, na_rep=na) if fmt else df.style.format(na_rep=na)
        st = st.set_table_styles(TABLE_STYLES)
        num = list(df.select_dtypes('number').columns)
        if num:
            st = st.set_properties(subset=num, **{'text-align': 'right'})
        for col, side in (align or {}).items():
            if col in df.columns:
                st = st.set_properties(subset=[col],
                                       **{'text-align': side,
                                          'white-space': 'pre'})
        if caption:
            st = st.set_caption(caption)
        if not index:
            st = st.hide(axis='index')
        if not header:
            st = st.hide(axis='columns')
        _display(st)
    except Exception:                                   # noqa: BLE001
        # ノートブックの外（スクリプトから import したとき）でも読める形
        if caption:
            print(caption)
        print(df.to_string(index=index, header=header))
    return df


def grid(items, ncol=8, caption=''):
    """語の並びを ``ncol`` 列の表にして表示する。

    40 語を1行に流すと折り返しで読めない。列に切ると目で追える。
    順位が要るなら ``show()`` に順位列を付けた表を渡すこと。
    """
    items = [str(x) for x in items]
    rows = [items[i:i + ncol] for i in range(0, len(items), ncol)]
    rows = [r + [''] * (ncol - len(r)) for r in rows]
    t = pd.DataFrame(rows, columns=[f'_{i}' for i in range(ncol)])
    return show(t, caption=caption, header=False)


def work_rows(meta_df=None):
    """``work_stem`` からメタデータの行を引く辞書を作る。

    ``meta_df`` を省くと**分析対象外の行も含めた全件**から作る。
    表示用の名前は，分析から外した作品についても引けるほうがよい。

    **鍵の綴りに注意。** 青空文庫の作品 ID は索引では 0 埋めされていない
    （``1743``）が，本パイプラインのファイル名は6桁に 0 埋めしてある
    （``000119_001743``）。素朴に連結すると ``000119_1743`` となり，
    **1件も一致しない**。辞書は空振りしても例外を出さないので，
    誰の何だか分からないまま最後まで通ってしまう。両方の綴りを登録する。

    ``file_v1`` は増補 45 点では空である。``os.path.splitext(nan)`` は
    例外になるので，文字列であることを確かめてから使う。
    """
    if meta_df is None:
        meta_df = load_meta(analysis_only=False)
    d = {}
    for _, r in meta_df.iterrows():
        fv = r.get('file_v1')
        if isinstance(fv, str) and fv.strip():
            d[os.path.splitext(fv)[0]] = r
        pid = str(r.get('aozora_person_id') or '').strip()
        wid = str(r.get('aozora_work_id') or '').strip()
        if pid and wid and pid.lower() != 'nan' and wid.lower() != 'nan':
            for k in (f'{pid.zfill(6)}_{wid.zfill(6)}',
                      f'{pid}_{wid}', f'{pid.zfill(6)}_{wid}'):
                d[k] = r
    return d


def work_labels(meta_df=None, maxlen=12, with_year=False):
    """``work_stem`` → ``作者『作品』`` の対応表を返す。

    ``000119_001743`` と出されても誰の何だか分からない。距離の近い
    ペアを見るときに**どの作家のどの作品か**が分からなければ，
    「作家効果か時代効果か」という問いにそもそも答えられない。
    表示するときは必ずこれを通すこと。
    """
    out = {}
    for k, r in work_rows(meta_df).items():
        t = str(r.get('title_aozora') or '')
        lab = f"{r.get('author_ja', '?')}『{t[:maxlen]}』"
        if with_year and str(r.get('year_first') or '').strip():
            lab += f"({r['year_first']})"
        out[k] = lab
    return out


def attach_meta(df, cols, stem_col='work_stem', meta_df=None, quiet=False,
                fill_blank=True):
    """``df`` に足りないメタデータの列を，``work_stem`` から引いて補う。

    ``fill_blank=True``（既定）なら，**列はあるのに値が空**のセルも補う。
    列が無いより，列があって半分が空のほうが危ない。列が無ければ
    ``AttributeError`` で止まるが，値が空だと**図がそのまま描けてしまう**。
    2026-09-22 に 07 の突合が外れ，101 点のうち 62 点の ``year_first`` が
    空になった。図は描けたが，62 点が「初出年不明」の灰色で並んだ。
    値の空きも数えて報告し，ここで補えるものは補う。

    分析スクリプトの出力は，その分析に要る列しか書かない。
    ``09_doc2vec.py`` の ``work_vectors.csv`` に ``genre_main`` が無いのは
    その一例である。ノートブックで ``wv.genre_main`` と書けば
    ``AttributeError: 'DataFrame' object has no attribute 'genre_main'``
    になるが，**足りないのは列であって情報ではない**。
    メタデータ表には必ずあるのだから，ここで引いて補えばよい。

    出力 CSV の列構成に図の描画が依存するのは弱い。分析スクリプトを
    書き換えるたびに図が落ちる。図の側で「要る列を宣言して取りに行く」
    ほうが，どちらを先に走らせても通る。

    引けなかった列は空のまま残し ``[warn]`` を出す。図が落ちるより，
    「この軸は塗れなかった」と分かったうえで出るほうがよい。
    """
    df = df.copy()
    if stem_col not in df.columns:
        if not quiet:
            print(f'[warn] {stem_col} 列が無いので補完できない: {list(cols)}')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        return df

    rows = work_rows(meta_df)

    # **まず鍵が合っているかを見る。** 合っていなければ何も補えない。
    # 「1件も合わない」のはたいてい 0 埋めの綴り違いで，黙って通すと
    # 全部の軸が空のまま図になる。
    stems = df[stem_col].astype(str)
    found = stems.map(lambda s: s in rows)
    if not quiet and not found.all():
        n_miss = int((~found).sum())
        lv = 'FATAL' if found.sum() == 0 else 'warn '
        print(f'[{lv}] {stem_col} がメタデータと突合できない行が '
              f'{n_miss}/{len(df)} 件ある: '
              + '，'.join(stems[~found].head(4)))
        if found.sum() == 0:
            print('        **1件も合っていない。** 作品 ID の 0 埋めの'
                  '綴り違いを疑うこと（例 000119_1743 と 000119_001743）。')
            print('        この表を作ったスクリプトの鍵の作り方を直すこと。')

    def _blank(v):
        return v is None or str(v).strip().lower() in ('', 'nan', 'none')

    for c in cols:
        if c not in df.columns:
            vals = [(lambda r: '' if r is None or _blank(r.get(c))
                     else r.get(c))(rows.get(s)) for s in stems]
            df[c] = vals
            n = int(sum(1 for v in vals if str(v).strip()))
            if not quiet:
                mark = 'ok  ' if n == len(df) else 'warn'
                print(f'[{mark}] {c} をメタデータから補完: {n}/{len(df)} 件')
            continue

        if not fill_blank:
            continue
        # 列はある。空のセルだけを埋める。
        blank = df[c].map(_blank)
        if not blank.any():
            continue
        filled = 0
        vals = df[c].tolist()
        for i, (s, is_blank) in enumerate(zip(stems, blank)):
            if not is_blank:
                continue
            r = rows.get(s)
            if r is not None and not _blank(r.get(c)):
                vals[i] = r.get(c)
                filled += 1
        df[c] = vals
        if not quiet:
            left = int(sum(1 for v in vals if _blank(v)))
            mark = 'fix ' if left == 0 else 'warn'
            print(f'[{mark}] {c} は {int(blank.sum())}/{len(df)} 件が空だった'
                  f' → {filled} 件をメタデータから補完'
                  + ('' if left == 0 else f'（なお {left} 件が空）'))
            if left:
                print('        **その列で塗る図・集計は，この件数を'
                      '報告に書くこと。**')
    return df


def label_points(ax, xs, ys, texts, fontsize=8, color='#333333', pad=4,
                 leader='line', leader_min=13, crowd_r=26,
                 leader_color='#8a8a83'):
    """散布図の注記を，重ならない位置だけに置き，遠いものは引き出し線で結ぶ。

    素朴に ``ax.annotate(t, (x, y))`` と書くと，**注目すべき点ほど一箇所に
    固まる**ので注記が必ず重なって読めなくなる。文語標識の上位は
    どれも口語標識がほぼ 0 で，対数軸の右下隅に密集するのが典型である。

    そこで点の周囲を順に試し，他の注記とも他の点とも重ならず，かつ軸の
    内側に収まる位置があればそこに置く。どこにも置けない注記は**置かずに
    数だけ報告する**。読めない字を重ねるより，図の外（下の表）で番号から
    引くほうがよい。

    **離れた位置に置いた注記は，引き出し線で点と結ぶ。** 避けた結果として
    注記は点から離れるので，線が無いとどの点の名前なのか分からなくなる
    ——密集した領域では隣の点の名前だと読まれる。線があれば，遠くへ逃がす
    ことに副作用が無くなるので，**近くに空きが無い注記も置ける**ようになる
    （候補の輪を 24・30 ポイントまで広げてあるのはそのため）。

    ``leader``
        ``'line'``（既定）… 矢じりの無い細線で結ぶ。図版の慣例はこちら。
        6.5pt の文字に矢じりを付けると，印そのものを覆って点が読めなくなる
        ``'arrow'`` … 小さな矢じりを付ける
        ``'none'`` … 結ばない（従来どおり）
    ``leader_min``
        この距離（ポイント）より遠くに置いた注記を結ぶ。既定は 0，
        つまり**すべて結ぶ**。注記は必ず点から離れた位置に置かれるので，
        離れている以上「どの点の名前か」は線でしか確定しない。
        線を省くと，隣の点の名前だと読まれる余地が残る。
    ``crowd_r``
        注記の近くに**自分以外の点**がこの半径（ピクセル）内にあるかを
        見る。``leader_min`` を上げて線を減らしたときでも，
        紛れる相手が居る注記だけは必ず結ぶための保険である。

    表示座標で矩形の重なりを見るので，**軸の位置が確定してから**呼ぶ。
    ``fig.tight_layout()`` はこの関数より**前**に呼ぶこと（後で呼ぶと軸が
    動き，せっかく避けた位置がずれる）。戻り値は置けた注記の数。
    """
    from matplotlib.transforms import Bbox
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    texts = list(texts)
    if len(texts) == 0:
        return 0
    # **長さが違えば黙って切り詰めずに止める。** zip は短いほうに合わせるので，
    # 座標だけを絞り込んで名前を絞り忘れると，先頭から順に**別の作品の名前**が
    # 貼られた図が，何の警告も出さずに出来上がる。これがいちばん重い事故である。
    if not (len(xs) == len(ys) == len(texts)):
        raise ValueError(
            f'label_points: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'名前={len(texts)}）。座標と名前を同じ添字で絞り込むこと。'
            'たとえば P[pick,0] と組むのは names[pick] であって names ではない。')
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(renderer=ren)

    def pad_box(b, w=2.0, h=1.5):
        """注記の矩形に**絶対量の余白**を足す。

        倍率（``expanded(1.08, …)``）では足りない。1桁の数字は幅 8px ほど
        なので 8% は 0.6px にしかならず，隣り合う注記が触れるほど近くても
        「重なっていない」と判定される。**``21`` が「21」と読める**のは
        これが原因である。文字の大小によらず一定の余白を確保する。
        """
        return Bbox.from_extents(b.x0 - w, b.y0 - h, b.x1 + w, b.y1 + h)

    def bb_of(ann):
        # Annotation 自身の get_window_extent を使うこと。
        # Text.get_window_extent(ann, ...) を呼ぶと xy の位置が無視され，
        # xytext のオフセットを絶対座標と見た矩形が返って判定が壊れる。
        return ann.get_window_extent(renderer=ren)

    blocked = []
    for coll in ax.collections:
        try:
            for p in coll.get_offsets():
                px, py = ax.transData.transform(p)
                blocked.append(Bbox.from_bounds(px - pad, py - pad,
                                                2 * pad, 2 * pad))
        except Exception:                                    # noqa: BLE001
            pass

    # **まっすぐ真上・真下を先に試す。** 点の直上に中央揃えで置ければ，
    # それがいちばん素直で，引き出し線も要らない。横へずらすのは，
    # 直上が塞がっていたときの次善である。
    #
    # 横へずらす輪は 12 ポイントから始める。**線が線として見える長さを
    # 確保する**ため。8 ポイントに置くと引き出し線が3ピクセルの点にしか
    # ならず，汚れと区別がつかない。
    # **真上に置けなければ，まず真上へ逃がす。** 横へ逃がすと注記の左右の
    # 順序が点の順序と入れ替わり，引き出し線も交差する。真上に段を重ねる
    # 限り，x は動かないので順序は必ず保たれる。横へずらすのは最後。
    # **横のずらし幅は小さく取る。** 横へ 30 ポイントも動かすと，注記が
    # 隣の点の真上に乗り，引き出し線で結んでも読みにくい。真上に段を
    # 重ねるほうが先で（x が動かないので順序が保たれる），横は 8→18
    # ポイントの範囲に収める。
    CAND = [(0, 9), (0, -11), (0, 20), (0, -22), (0, 31), (0, -33),
            (8, 5), (-8, 5), (8, -12), (-8, -12),
            (11, 0), (-11, 0),
            (13, 9), (-13, 9), (13, -16), (-13, -16),
            (18, 0), (-18, 0), (18, 14), (-18, 14),
            (0, 42), (0, -44)]

    def _ha(dx):
        # dx が 0 なら**中央揃え**。ここを 'left' にすると，真上に置いた
        # つもりの注記が文字幅の半分だけ右にずれ，隣の点の上に乗る。
        return 'center' if dx == 0 else ('left' if dx > 0 else 'right')
    placed, chosen, skipped = [], [], 0
    for x, y, t in zip(xs, ys, texts):
        # **自分が指している点は避けない。** 除かないと，注記は必ず
        # 自分の点の隣に来るので全部「重なる」と判定され，1つも置けない。
        ox, oy = ax.transData.transform((x, y))
        near = [b for b in blocked
                if not (abs((b.x0 + b.x1) / 2 - ox) < 1
                        and abs((b.y0 + b.y1) / 2 - oy) < 1)]
        for dx, dy in CAND:
            ann = ax.annotate(str(t), (x, y), textcoords='offset points',
                              xytext=(dx, dy), fontsize=fontsize, color=color,
                              ha=_ha(dx),
                              va='bottom' if dy >= 0 else 'top', zorder=6)
            bb = pad_box(bb_of(ann))
            inside = (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                      and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1)
            if inside and not any(bb.overlaps(b) for b in placed + near):
                placed.append(bb)
                chosen.append((ann, x, y, str(t), dx, dy))
                break
            ann.remove()
        else:
            skipped += 1

    # ---- 交差をほどく ----------------------------------------------------
    # **引き出し線が交差すると，注記の左右の順序が点の順序と入れ替わる。**
    # 文語標識の上位のように順位そのものが意味を持つ図では，2 と 3 が
    # 入れ替わって並ぶだけで読み違えられる。交差している2件を見つけ，
    # **位置を入れ替えて交差が解ければ入れ替える**（2-opt）。
    def _cross(p, q, r, s):
        def o(a, b, c):
            return ((b[0] - a[0]) * (c[1] - a[1])
                    - (b[1] - a[1]) * (c[0] - a[0]))
        return (((o(r, s, p) > 0) != (o(r, s, q) > 0))
                and ((o(p, q, r) > 0) != (o(p, q, s) > 0)))

    kpt = fig.dpi / 72.0

    def _seg(i):
        ann, x, y, t, dx, dy = chosen[i]
        ox, oy = ax.transData.transform((x, y))
        return (ox, oy), (ox + dx * kpt, oy + dy * kpt)

    def _set_off(i, dx, dy):
        ann, x, y, t, _, _ = chosen[i]
        ann.set_position((dx, dy))
        ann.set_ha(_ha(dx))
        ann.set_va('bottom' if dy >= 0 else 'top')
        chosen[i] = (ann, x, y, t, dx, dy)

    def _fits(i, bb):
        ox, oy = ax.transData.transform((chosen[i][1], chosen[i][2]))
        if not (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1):
            return False
        return not any(bb.overlaps(b) for b in blocked
                       if abs((b.x0 + b.x1) / 2 - ox) > 1
                       or abs((b.y0 + b.y1) / 2 - oy) > 1)

    swaps = 0
    for _ in range(3):
        improved = False
        for i in range(len(chosen)):
            for j in range(i + 1, len(chosen)):
                if not _cross(*_seg(i), *_seg(j)):
                    continue
                di, dj = chosen[i][4:6], chosen[j][4:6]
                _set_off(i, *dj)
                _set_off(j, *di)
                bi = pad_box(bb_of(chosen[i][0]))
                bj = pad_box(bb_of(chosen[j][0]))
                others = [b for k2, b in enumerate(placed) if k2 not in (i, j)]
                good = (not bi.overlaps(bj)
                        and not any(bi.overlaps(b) or bj.overlaps(b)
                                    for b in others)
                        and _fits(i, bi) and _fits(j, bj)
                        and not _cross(*_seg(i), *_seg(j)))
                if good:
                    placed[i], placed[j] = bi, bj
                    swaps += 1
                    improved = True
                    continue
                _set_off(i, *di)
                _set_off(j, *dj)

                # 入れ替えが収まらないときは，**片方を別の候補位置へ動かす**。
                # 入れ替えは2つの箱の大きさが違うと失敗しやすい（数字1桁と
                # 作者名では幅が違う）。動かすほうは箱の大きさが変わらない。
                moved = False
                for who, other in ((i, j), (j, i)):
                    d0 = chosen[who][4:6]
                    for cx, cy in CAND:
                        if (cx, cy) == tuple(d0):
                            continue
                        _set_off(who, cx, cy)
                        bw = pad_box(bb_of(chosen[who][0]))
                        rest = [b for k2, b in enumerate(placed) if k2 != who]
                        if (_fits(who, bw)
                                and not any(bw.overlaps(b) for b in rest)
                                and not _cross(*_seg(who), *_seg(other))
                                and not any(_cross(*_seg(who), *_seg(k2))
                                            for k2 in range(len(chosen))
                                            if k2 != who)):
                            placed[who] = bw
                            swaps += 1
                            moved = improved = True
                            break
                        _set_off(who, *d0)
                    if moved:
                        break
        if not improved:
            break

    # ---- 引き出し線 ------------------------------------------------------
    # **線は配置が全部決まってから付ける。** arrowprops を付けた
    # Annotation の get_window_extent は「文字＋線」の外接矩形を返すので，
    # 配置の判定に使うと自分の点と必ず重なり，1件も置けなくなる。
    n_leader = 0
    if leader in ('line', 'arrow'):
        style = '-' if leader == 'line' else '-|>'
        for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
            ox, oy = ax.transData.transform((x, y))

            def dist_to_box(px, py, b=bb):
                # 文字の矩形から点までの距離。矩形の中なら 0。
                ddx = max(b.x0 - px, 0, px - b.x1)
                ddy = max(b.y0 - py, 0, py - b.y1)
                return (ddx * ddx + ddy * ddy) ** .5

            # **素直に置けたものには線を引かない。** 点の直上（または直下）に
            # 中央揃えで載っていて，しかもその注記にいちばん近い点が自分の
            # 点であれば，どの点の名前かは見れば分かる。線はかえって邪魔
            # である。横へ逃がしたものだけを結ぶ。
            if dx == 0 and abs(dy) <= 12:
                continue            # 点の真上・真下の一段目 → 線は要らない
            # それ以外は結ぶ。**段を上げたものも結ぶ。** 一段上げた注記の
            # 真下には別の点の注記が入るので，どちらの点のものか分からなく
            # なる。横へずらしたものは言うまでもない。
            d_other = min(
                (dist_to_box((b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2)
                 for b in blocked
                 if abs((b.x0 + b.x1) / 2 - ox) > 1
                 or abs((b.y0 + b.y1) / 2 - oy) > 1),
                default=float('inf'))
            if (dx * dx + dy * dy) ** .5 < leader_min and d_other >= crowd_r:
                continue
            ann.remove()
            ax.annotate(t, (x, y), textcoords='offset points',
                        xytext=(dx, dy), fontsize=fontsize, color=color,
                        ha=_ha(dx),
                        va='bottom' if dy >= 0 else 'top', zorder=6,
                        arrowprops=dict(arrowstyle=style, linewidth=.55,
                                        color=leader_color, alpha=.9,
                                        shrinkA=1.5, shrinkB=2.5,
                                        mutation_scale=7))
            n_leader += 1

    # ---- 誤読の自己点検 --------------------------------------------------
    # **注記の最寄りの点が自分の点でないものを数える。** これが
    # 「ラベルとデータ点がずれて見える」の正体である。引き出し線を
    # 引いてあれば誤読にはならないが，線を切った設定では危険なので，
    # そのときだけ警告を出す。
    risky = []
    for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
        ox, oy = ax.transData.transform((x, y))
        cx, cy = (bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2
        d_own = ((cx - ox) ** 2 + (cy - oy) ** 2) ** .5
        d_other = min(
            (((b.x0 + b.x1) / 2 - cx) ** 2 + ((b.y0 + b.y1) / 2 - cy) ** 2) ** .5
            for b in blocked
            if abs((b.x0 + b.x1) / 2 - ox) > 1 or abs((b.y0 + b.y1) / 2 - oy) > 1
        ) if len(blocked) > 1 else float('inf')
        if d_other < d_own:
            risky.append(t)
    left = sum(1 for i in range(len(chosen)) for j in range(i + 1, len(chosen))
               if _cross(*_seg(i), *_seg(j)))
    if skipped:
        print(f'[fig] 重なるため {skipped} 件の注記を省いた（表で引くこと）')
    if left:
        print(f'[warn] 引き出し線の交差が {left} 件ほどけなかった。'
              '注記の左右の順序が点の順序と食い違う。'
              '注記を短くするか，件数を減らすこと。')
    if risky:
        head = '，'.join(str(r) for r in risky[:6])
        more = f' ほか{len(risky) - 6}件' if len(risky) > 6 else ''
        if leader in ('line', 'arrow'):
            print(f'[fig] {len(risky)} 件の注記は別の点のほうが近い'
                  f'（{head}{more}）。引き出し線で結んであるので読み違えない。')
        else:
            print(f'[warn] {len(risky)} 件の注記は**別の点のほうが近い**'
                  f'（{head}{more}）。leader="none" では読み違えが起きる。')
    return len(placed)
def reserve_right(fig, frac=0.80):
    """面の外に凡例を置いた図で，**右に余白を確保する**。

    ``tight_layout()`` は面の外に置いた凡例を数えないので，そのままだと
    凡例が図の枠から出る。静止版は ``bbox_inches='tight'`` で救われるが，
    **HTML に埋め込む版は切り取らない**（切り取ると点の位置の割合が
    ずれる）ので，凡例が切れて読めなくなる。実際に切れた。

    ``tight_layout()`` の**後**，注記（``label_points``）の**前**に呼ぶ。
    """
    fig.subplots_adjust(right=frac)


def save_fig(fig, stem, out=None):
    """図を SVG で保存してパスを表示する。

    stem は拡張子なしの名前（例 'Step1_period_balance'）。
    点が数千個ある散布図は，散布図だけ rasterized=True にしておくと
    軸と文字はベクタのままファイルが軽くなる。
    """
    d = Path(out) if out else OUT
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.{FIG_EXT}'
    fig.savefig(path, format=FIG_EXT, dpi=RASTER_DPI, bbox_inches='tight')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB)')
    return path


# ---------------------------------------------------------------------------
# 対話的な図（SVG はそのまま残す）
# ---------------------------------------------------------------------------
# 散布図の点が何百個あると，注記を付けられるのはごく一部である。残りの点は
# 「どの語か」が分からないまま眺めることになる。かといって全点に名前を
# 付ければ図は読めない。
#
# そこで**同じ図から2つ出す**。
#   * ``<stem>.svg``  … 論文・配布用。これまでどおり。加筆も拡大も自由
#   * ``<stem>.html`` … 授業・探索用。SVG をそのまま埋め込み，
#                       その上に当たり判定を重ねて，指した点の語を出す
#
# **HTML は SVG を作り直さない。同じ SVG を中に入れる。** 別に描き直すと
# 図が2種類できて，どちらが正かが分からなくなる。注記（bursty な語の
# ラベル）も SVG の中にあるのでそのまま残る。
#
# 外部の JS ライブラリは使わない。CDN が塞がれた機体でも開けるようにする。
INTERACTIVE_CSS = """
:root { --ink:#1f1e1b; --ink2:#5a5a55; --line:#d8d7d0; --surface:#ffffff;
        --wash:#f7f7f4; --accent:#184f95; }
* { box-sizing:border-box; }
body { margin:0; padding:24px 16px 48px; background:var(--wash);
       color:var(--ink); font-family:"Hiragino Sans","Noto Sans JP",
       "Yu Gothic",system-ui,sans-serif; line-height:1.6; }
.wrap { max-width:1100px; margin:0 auto; }
h1 { font-size:1.15rem; margin:0 0 .2em; font-weight:650; }
.sub { color:var(--ink2); font-size:.86rem; margin:0 0 1.1em; }
.card { background:var(--surface); border:1px solid var(--line);
        border-radius:10px; padding:14px; }
.figbox { position:relative; }
.figbox svg { width:100%; height:auto; display:block; }
#hit { position:absolute; inset:0; cursor:crosshair; }
#ring { position:absolute; width:22px; height:22px; margin:-11px 0 0 -11px;
        border:2px solid var(--accent); border-radius:50%;
        pointer-events:none; opacity:0; transition:opacity .08s; }
#tip { position:absolute; z-index:5; min-width:190px; max-width:290px;
       background:var(--surface); border:1px solid var(--line);
       border-radius:8px; box-shadow:0 6px 20px rgba(0,0,0,.13);
       padding:9px 11px; font-size:.8rem; pointer-events:none; opacity:0;
       transition:opacity .08s; }
#tip .term { font-size:1.05rem; font-weight:650; letter-spacing:.02em;
             margin-bottom:.35em; word-break:break-all; }
#tip dl { display:grid; grid-template-columns:auto 1fr; gap:1px 10px;
          margin:0; }
#tip dt { color:var(--ink2); font-size:.74rem; white-space:nowrap; }
#tip dd { margin:0; text-align:right; font-variant-numeric:tabular-nums;
          font-weight:600; }
.bar { display:flex; gap:10px; align-items:center; flex-wrap:wrap;
       margin:14px 0 0; font-size:.82rem; color:var(--ink2); }
.bar input { font:inherit; padding:5px 9px; border:1px solid var(--line);
             border-radius:6px; min-width:190px; background:var(--surface); }
.bar a { color:var(--accent); }
table { border-collapse:collapse; width:100%; font-size:.78rem;
        margin-top:10px; }
th,td { padding:4px 8px; border-bottom:1px solid #ecebe6; text-align:left;
        white-space:nowrap; }
th { background:var(--wash); position:sticky; top:0; font-weight:650; }
td.num { text-align:right; font-variant-numeric:tabular-nums; }
tbody tr:hover td { background:var(--wash); }
tbody tr.on td { background:#eaf1fb; }
.scroll { max-height:340px; overflow:auto; border:1px solid var(--line);
          border-radius:8px; margin-top:10px; }
.hint { font-size:.78rem; color:var(--ink2); margin:.6em 0 0; }
#links { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#links line { stroke:var(--accent); stroke-width:1.1; opacity:.55; }
#links circle { fill:none; stroke:var(--accent); stroke-width:1.4; opacity:.8; }
#marks { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#marks circle { fill:none; stroke:#d55e00; stroke-width:1.6; opacity:.9; }
#tip .notes { margin:.45em 0 0; font-size:.76rem; color:var(--ink);
              border-top:1px solid var(--line); padding-top:.4em;
              line-height:1.5; word-break:break-all; }
#tip .notes b { color:var(--ink2); font-weight:600; }
.prov { font-size:.72rem; color:var(--ink2); margin:.9em 0 0;
        border-top:1px solid var(--line); padding-top:.6em;
        font-variant-numeric:tabular-nums; }
.danger { background:#fdf0ea; border:1px solid #e8a37c; border-radius:8px;
          padding:9px 12px; font-size:.85rem; color:#8a3b10;
          margin:0 0 12px; }
"""

INTERACTIVE_JS = r"""
// 点は data-* ではなく JSON で渡す。語はコーパス由来の任意の文字列なので，
// **HTML に文字列連結で差し込まない**（textContent で入れる）。
// 見出し（keys・nhead）は全点で同じなら1回だけ入っている。点が1万個ある
// 図では，これで HTML が 1 MB 以上軽くなる。古い形（配列だけ）も読む。
const RAW = JSON.parse(document.getElementById('pts-data').textContent);
const PTS = Array.isArray(RAW) ? RAW : RAW.pts;
const KEYS = (RAW && RAW.keys) || [];
const NHEAD = (RAW && RAW.nhead) || '';
const LINKNOTES = !!(RAW && RAW.linknotes);
function pairsOf(p) {
  if (p.fields) return p.fields;
  if (p.v) return p.v.map((x, i) => [KEYS[i] || '', x]);
  return [];
}
function notesOf(p) {
  if (p.notes) return p.notes;
  if (p.n) return [NHEAD, p.n];
  // 本文が無く linknotes が立っているときは，線で結ぶ先の語を並べる
  if (LINKNOTES && p.links && p.links.length) {
    return [NHEAD, p.links.map(j => (PTS[j] || {}).term || '').join(' ')];
  }
  return null;
}
const box = document.getElementById('hit');
const tip = document.getElementById('tip');
const ring = document.getElementById('ring');
const rows = Array.from(document.querySelectorAll('tbody tr'));
const links = document.getElementById('links');
const marks = document.getElementById('marks');

// **最も近い点を拾う。** 点の直径は数ピクセルしかないので，
// 「真上に置く」ことを要求すると誰も当てられない（dataviz の規則）。
// カーソルに最も近い点を選び，遠すぎるときだけ何も出さない。
function nearest(px, py, w, h) {
  let best = null, bd = 1e9;
  for (const p of PTS) {
    const dx = p.x * w - px, dy = p.y * h - py;
    const d = dx * dx + dy * dy;
    if (d < bd) { bd = d; best = p; }
  }
  return Math.sqrt(bd) <= 34 ? best : null;   // 34px より遠ければ出さない
}

function fill(p) {
  tip.textContent = '';
  const h = document.createElement('div');
  h.className = 'term';
  h.textContent = p.term;                     // ← 連結しない
  tip.appendChild(h);
  const dl = document.createElement('dl');
  for (const [k, v] of pairsOf(p)) {
    const dt = document.createElement('dt'); dt.textContent = k;
    const dd = document.createElement('dd'); dd.textContent = v;
    dl.appendChild(dt); dl.appendChild(dd);
  }
  tip.appendChild(dl);
  const nt = notesOf(p);
  if (nt) {                                   // 近傍語など，横に長い情報
    const n = document.createElement('p');
    n.className = 'notes';
    const b = document.createElement('b');
    b.textContent = nt[0] + ' ';
    n.appendChild(b);
    n.appendChild(document.createTextNode(nt[1]));
    tip.appendChild(n);
  }
}

// **原空間での近傍を線で結ぶ。** 画面の近さは射影の結果にすぎない。
// 線が遠くへ伸びるなら，その点の近傍関係は2次元に収まっていない。
// これを見せるのが，この図でいちばん大事なところである。
function drawLinks(p, w, h) {
  if (!links) return;
  while (links.firstChild) links.removeChild(links.firstChild);
  if (!p.links || !p.links.length) return;
  const NS = 'http://www.w3.org/2000/svg';
  for (const j of p.links) {
    const q = PTS[j];
    if (!q) continue;
    const ln = document.createElementNS(NS, 'line');
    ln.setAttribute('x1', p.x * w); ln.setAttribute('y1', p.y * h);
    ln.setAttribute('x2', q.x * w); ln.setAttribute('y2', q.y * h);
    links.appendChild(ln);
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', q.x * w); c.setAttribute('cy', q.y * h);
    c.setAttribute('r', 5);
    links.appendChild(c);
  }
}

let cur = null;
function show(p, px, py) {
  const w = box.clientWidth, h = box.clientHeight;
  if (p !== cur) { fill(p); drawLinks(p, w, h); cur = p; }
  ring.style.left = (p.x * w) + 'px';
  ring.style.top = (p.y * h) + 'px';
  ring.style.opacity = 1;
  tip.style.opacity = 1;
  // はみ出さないように寄せる
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lx = px + 16, ly = py + 14;
  if (lx + tw > w) lx = px - tw - 16;
  if (ly + th > h) ly = py - th - 14;
  tip.style.left = Math.max(0, lx) + 'px';
  tip.style.top = Math.max(0, ly) + 'px';
  rows.forEach(r => r.classList.toggle('on', r.dataset.i === String(p.r)));
}
function hide() {
  tip.style.opacity = 0; ring.style.opacity = 0; cur = null;
  if (links) while (links.firstChild) links.removeChild(links.firstChild);
  rows.forEach(r => r.classList.remove('on'));
}

box.addEventListener('pointermove', e => {
  const r = box.getBoundingClientRect();
  const p = nearest(e.clientX - r.left, e.clientY - r.top, r.width, r.height);
  if (p) show(p, e.clientX - r.left, e.clientY - r.top); else hide();
});
box.addEventListener('pointerleave', hide);

// 表の行にカーソルを乗せても，図の上の点が光る（逆引き）。
// **カーソルが使えない人にも同じ情報が届くように**，表を必ず添える
// （点が数千を超える図だけは表を絞る。絞ったことは図の下に明記する）。
rows.forEach(r => {
  r.addEventListener('mouseenter', () => {
    // 表の行は論理点。図の上では**先頭の面**の点を光らせる
    const p = PTS[Number(r.dataset.i)];
    if (!p) return;
    const w = box.clientWidth, h = box.clientHeight;
    show(p, p.x * w, p.y * h);
  });
  r.addEventListener('mouseleave', hide);
});

// 絞り込み。語・作品・時代のどれでも当たる
const q = document.getElementById('q');
if (q) q.addEventListener('input', () => {
  const s = q.value.trim();
  let n = 0;
  rows.forEach(r => {
    const hit = !s || r.textContent.includes(s);
    r.style.display = hit ? '' : 'none';
    if (hit) n++;
  });
  // 表を絞った図では，**表に無い語も図の上では当たる**。
  // 表の件数だけを出すと「無い」と誤解されるので両方を出す。
  const nlog = Number(document.body.dataset.nlog || rows.length);
  let extra = '';
  if (s && rows.length < nlog) {
    const seen = new Set();
    for (const p of PTS) if (p.term.includes(s)) seen.add(p.r);
    extra = '（図の上 ' + seen.size + ' 件）';
  }
  document.getElementById('count').textContent = n + ' 件' + extra;
  // **図の上にも印を付ける。** 表だけ絞っても「どこにあるか」は分からない。
  if (!marks) return;
  while (marks.firstChild) marks.removeChild(marks.firstChild);
  if (!s) return;
  const NS = 'http://www.w3.org/2000/svg';
  const w = box.clientWidth, h = box.clientHeight;
  let drawn = 0;
  for (const p of PTS) {
    if (!p.term.includes(s)) continue;
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', p.x * w); c.setAttribute('cy', p.y * h);
    c.setAttribute('r', 7);
    marks.appendChild(c);
    if (++drawn > 400) break;          // 印が多すぎると図が読めない
  }
});
"""


def save_interactive(fig, ax, stem, xs, ys, tips, out=None, title='',
                     note='', table_cols=None, source=None, id_col='語',
                     hint='', table_idx=None):
    """SVG を保存し，**同じ SVG を埋め込んだ対話的な HTML** も書く。

    ``xs`` ``ys`` はデータ座標，``tips`` は点ごとの情報
    （``{'term': 語, 'fields': [(見出し, 値), …]}`` の並び）。
    3つの長さは一致していなければならない。ずれたまま描くと，
    **指した点と出る語が食い違う**（注記の添字ずれと同じ事故）。

    位置は「図全体に対する割合」で書き出す。SVG を ``width:100%`` で
    伸縮させても割合は変わらないので，どんな幅でも点と当たり判定が
    合う。座標は matplotlib の変換を通して得るので，**図と HTML で
    座標の計算が二重にならない**。

    ``source`` に入力ファイルのパスを渡すこと。**どの表から描いた図かを
    HTML の末尾に刻む。** これが無いと，試験用の作りかけのデータから
    描いた図と，本番のデータから描いた図が見分けられない。
    入力がプロジェクトの外（``/tmp`` など）にあるときは
    「試験用」と赤字で出し，配布してはいけないことを図自身に言わせる。

    ``id_col`` は表の第1列の見出し（既定「語」。作品を点にする図では
    「作品」などに変える）。

    ``ax`` には**面の並び**も渡せる（``[axes[0], axes[1]]``）。同じ点を
    別の塗り分けで2面に描いた図では，どちらの面を指しても同じ情報が出る。
    表の行は点ごとに1行だけ作る（面の数だけ重複させない）。

    ``table_idx`` は**表に載せる点の添字**（既定は全点）。点が数千を超える
    図では表を全件出すと HTML が数 MB になり，読む側にも役に立たない。
    そのときは載せる点を選ぶ。**ただし図の当たり判定と検索は全点に効く**
    ので，表に無い語も指せるし検索で図に印が付く。表を絞ったときは，
    何件のうち何件を載せたかを HTML に明記する（黙って捨てないこと）。
    """
    import json as _json
    if not (len(xs) == len(ys) == len(tips)):
        raise ValueError(
            f'save_interactive: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'情報={len(tips)}）。座標と情報を同じ添字で絞り込むこと。')

    svg_path = save_fig(fig, stem, out=out)
    outdir = svg_path.parent

    # ---- 埋め込む SVG は**切り取らずに**保存する ------------------------
    # save_fig は bbox_inches='tight' で余白を詰めるため，図全体に対する
    # 割合と，ファイルの座標系がずれる。埋め込み用は詰めずに出す。
    import io
    buf = io.StringIO()
    fig.savefig(buf, format='svg', dpi=RASTER_DPI)
    svg = buf.getvalue()
    svg = svg[svg.index('<svg'):]          # XML 宣言と DOCTYPE を落とす

    # ---- 点の位置を図全体に対する割合で得る -----------------------------
    axes_list = list(ax) if isinstance(ax, (list, tuple, np.ndarray)) else [ax]
    W, H = fig.bbox.width, fig.bbox.height
    n_pts = len(tips)

    # **見出しは点ごとに書かない。** 点が1万個ある図では，
    # 「品詞」「頻度」…という見出しを1万回繰り返すだけで HTML が
    # 1 MB 以上ふくらむ。全点で見出しが同じなら1回だけ書き，
    # 値の並びだけを点に持たせる（JS 側で組み直す）。
    keys = [str(k) for k, _ in tips[0].get('fields', [])] if tips else []
    same_keys = bool(keys) and all(
        [str(k) for k, _ in t.get('fields', [])] == keys for t in tips)
    nheads = {str(t['notes'][0]) for t in tips if t.get('notes')}
    nhead = next(iter(nheads)) if len(nheads) == 1 else ''
    # 本文を渡さず ``notes=(見出し, None)`` としたときは，
    # ``links`` の先の語を JS 側で並べる
    linknotes = bool(nhead) and any(
        t.get('notes') and t['notes'][1] is None and t.get('links')
        for t in tips)

    pts = []
    for k, axk in enumerate(axes_list):
      pxy = axk.transData.transform(np.column_stack([np.asarray(xs, float),
                                                     np.asarray(ys, float)]))
      for j, (t, (px, py)) in enumerate(zip(tips, pxy)):
        i = k * n_pts + j
        # **変数名に注意。** ここを d と書くと，上で取った出力先 d
        # （svg_path.parent）を上書きして，最後に d / '....html' が
        # 「dict ÷ str」になる。実際に踏んだ。名前は使い回さない。
        # 添字 i は JS では使わない（行は r で引く）。点が1万個ある図では
        # 使わない値も 100 KB 単位で効くので書かない。
        rec = {'r': j, 'term': str(t.get('term', '')),
               'x': round(float(px) / W, 6),
               'y': round(1 - float(py) / H, 6)}        # SVG は上が 0
        if same_keys:
            rec['v'] = [str(b) for _, b in t.get('fields', [])]
        else:
            rec['fields'] = [[str(a), str(b)] for a, b in t.get('fields', [])]
        if t.get('notes'):
            # ('見出し', '本文') の2つ組。横に長い情報（近傍語など）。
            # 本文を None にすると，**線で結ぶ先の語を JS が並べる**
            # （同じ語の列を点ごとに書かずに済む。1万点で 1 MB 近く効く）
            if t['notes'][1] is None:
                pass
            elif nhead:
                rec['n'] = str(t['notes'][1])
            else:
                rec['notes'] = [str(t['notes'][0]), str(t['notes'][1])]
        if t.get('links'):
            # 原空間での近傍の添字。**同じ面の中で**線を結ぶ
            rec['links'] = [k * n_pts + int(q) for q in t['links']]
        pts.append(rec)

    cols = table_cols or keys
    head = f'<tr><th>{_esc(id_col)}</th>' + ''.join(
        f'<th>{_esc(c)}</th>' for c in cols) + '</tr>'
    # 数字の列だけ右寄せにする。時代名や作品 ID を右寄せにすると読みにくい。
    def _numish(v):
        t = str(v).strip().replace('%', '').replace(',', '')
        t = t.lstrip('+-')
        return bool(t) and t.replace('.', '', 1).isdigit()

    # 表に載せる点。**面の数だけ重複させない**（論理点1つに1行）
    if table_idx is None:
        order = list(range(n_pts))
    else:
        seen, order = set(), []
        for i in [int(q) for q in table_idx]:   # 重複を除きつつ順序は保つ
            if 0 <= i < n_pts and i not in seen:
                seen.add(i); order.append(i)

    # 表は **tips から作る**（点の JSON は見出しを省いてあるので）
    body = []
    for j in order:
        fv = {str(a): str(b) for a, b in tips[j].get('fields', [])}
        tds = ''
        for c in cols:
            v = fv.get(c, '')
            cls = ' class="num"' if _numish(v) else ''
            tds += f'<td{cls}>{_esc(v)}</td>'
        body.append(f'<tr data-i="{j}">'
                    f'<td>{_esc(tips[j].get("term", ""))}</td>{tds}</tr>')

    # ---- 由来を図自身に刻む -------------------------------------------
    # **どの表から描いた図かが分からないと，試験用のデータで描いた図が
    # 本物として配られる。** 実際に起きた（2026-09-22）。
    import datetime as _dt
    stamp = _dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')
    src = Path(source) if source else None
    prov = f'点 {len(pts)} 個／作図 {stamp}'
    warn = ''
    if src is not None:
        try:
            mt = _dt.datetime.fromtimestamp(src.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        except OSError:
            mt = '不明'
        prov = f'入力 {src.name}（更新 {mt}）／' + prov
        # プロジェクトの外（/tmp など）から描いた図は試験用である
        try:
            outside = not str(src.resolve()).startswith(str(ROOT.resolve()))
        except Exception:                               # noqa: BLE001
            outside = True
        if outside or '/tmp/' in str(src):
            warn = ('<p class="danger">⚠ <b>試験用の入力から作った図である。'
                    f'配布してはいけない。</b>（入力 {_esc(str(src))}）</p>')
            prov = f'入力 {_esc(str(src))}／' + f'点 {len(pts)} 個／作図 {stamp}'

    html = f"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title or stem)}</title>
<style>{INTERACTIVE_CSS}</style></head>
<body data-nlog="{n_pts}" data-ntable="{len(body)}"><div class="wrap">
<h1>{_esc(title or stem)}</h1>
<p class="sub">{_rich(note)}</p>
{warn}
<div class="card">
  <div class="figbox">
    {svg}
    <svg id="links"></svg><svg id="marks"></svg>
    <div id="hit"></div><div id="ring"></div><div id="tip"></div>
  </div>
  <p class="hint">{_rich(hint or '点にカーソルを近づけると語が出る（最も近い点を拾うので，真上に置かなくてよい）。図の中の注記は静止版と同じものである。')}</p>
  <div class="bar">
    <input id="q" type="search" placeholder="語・作品・時代で絞り込む">
    <span id="count">{len(body)} 件</span>
    <span>·</span>
    {(f'<span>表は {len(body)} 件（図の点は {n_pts} 件。'
      '表に無い語も図の上で指せる。検索は図の印にも効く）</span><span>·</span>')
     if len(body) < n_pts else ''}
    <a href="{_esc(svg_path.name)}" download>SVG を保存</a>
    <span>（この HTML の中の図はその SVG そのもの）</span>
  </div>
  <div class="scroll"><table><thead>{head}</thead>
    <tbody>{''.join(body)}</tbody></table></div>
  <p class="prov">{prov}</p>
</div>
<script type="application/json" id="pts-data">{_json.dumps(
    {'keys': keys if same_keys else [], 'nhead': nhead,
     'linknotes': linknotes, 'pts': pts},
    ensure_ascii=False, separators=(',', ':'))}</script>
<script>{INTERACTIVE_JS}</script>
</div></body></html>
"""
    path = outdir / f'{stem}.html'
    path.write_text(html, encoding='utf-8')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB・対話版／'
          f'点 {len(pts)} 個)')
    return svg_path, path


def _esc(s):
    """HTML の特殊文字を落とす。**語はコーパス由来なので必ず通す。**"""
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _rich(s):
    """注記の ``**…**`` だけを太字にする。

    説明文をノートブックと同じ書き方（Markdown 風）で書けるようにする。
    **先に必ず _esc を通す**ので，タグを書き込まれる余地は無い。
    ``**`` のままだと HTML では記号がそのまま出て読みにくい。
    """
    import re as _re
    return _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', _esc(s))


PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
           '#56B4E9', '#D55E00', '#F0E442', '#666666']

# --- 順序のあるものを塗るための1色相のランプ -----------------------------
# **時代・年次・段階のように順序のあるものを，上の8色で塗ってはいけない。**
# 明治中期が青で明治後期が黄なら，隣り合う時代が隣り合う色にならず，
# 「時代が下るとどちらへ動くか」という肝心のことが読めなくなる。
# 1色相の濃淡にすれば，近いもの同士が近い色になり，勾配がそのまま見える。
# 散布図の点は白地の上に置くので，いちばん明るい段は 100 ではなく
# 250（背景との対比 2:1）から始める。100 は面で塗るとき（ヒートマップ）用。
SEQ_BLUE_STEPS = ['#86b6ef', '#5598e7', '#3987e5',
                  '#256abf', '#184f95', '#0d366b']
SEQ_BLUE = LinearSegmentedColormap.from_list('jlit_blue', SEQ_BLUE_STEPS)

# 離散の順序（4区分など）を塗るときはこちら。隣の段と明度差が十分あり，
# いちばん明るい段も背景から浮く（対比 2:1 以上）ことを確かめてある。
SEQ_BLUE_5 = ['#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b']

# 大分類の2色。散布図ではどの2点も隣り合いうるので**全ペアが
# 見分けられる必要**があり，使える色数は多くない。2色に絞って，
# 下位の区別は印の形に持たせる。
GENRE_C = {'Fiction': '#2a78d6', 'Nonfiction': '#eb6834'}

# --- 初出年の5段 ---------------------------------------------------------
# 切れ目は period と同じ 1900／1912／1926／1945。
# **6段にはできない。** 1色相の濃淡で順序を見せるには，隣り合う段の明度差が
# 0.06 以上要る。この青系ランプは 250→700 で明度差にして 0.30 ほどしか幅が
# 無いので，段を6つ取るとどこかが 0.05 台に落ち，隣が見分けられなくなる。
# そこで作品数3点の明治前期（〜1886）を明治中期にまとめて5段とする。
YEAR_EDGES = [1900, 1912, 1926, 1945]
YEAR_LABELS = ['〜1899 明治前・中期', '1900-1911 明治後期', '1912-1925 大正',
               '1926-1944 昭和戦前', '1945- 昭和戦後']


def year_bands(years):
    """初出年を5段に畳み，``(段番号, ラベル, 色)`` を返す。

    段番号は 0〜4。**初出年が読めないものは -1** にする。0 に落とすと
    年の分からない作品が全部いちばん古い段に入り，通時の議論が崩れる。

    時代で塗る図はすべてこれを通すこと。同じ色が全ステップで同じ時代を
    指すようになり，Step 1 の図と Step 7 の図を並べて読める。
    """
    y = pd.to_numeric(pd.Series(list(years)), errors='coerce')
    code = np.full(len(y), -1, dtype=int)
    ok = y.notna().values
    if ok.any():
        code[ok] = np.digitize(y[ok].values, YEAR_EDGES)
    return code, YEAR_LABELS, SEQ_BLUE_5


def run_script(script, *args, tail=4000):
    """scripts/ のスクリプトを実行し，標準出力・標準エラー・終了コードを必ず表示する。

    print(r.stdout or r.stderr) では，標準出力が空でないときに
    エラーの内容が隠れてしまう。学習用には両方見えるほうがよい。
    """
    cmd = [sys.executable, str(ROOT / 'scripts' / script)] + [str(a) for a in args]
    print('$ python', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-tail:])
    if r.stderr.strip():
        print('--- stderr ---')
        print(r.stderr[-tail:])
    print(f'[exit {r.returncode}]')
    return r


# 自分の作業フォルダ（共用 iMac では必ず自分の名前で作ること）
ME = os.environ.get('JLIT_USER', 'student')
# 使うメタデータ。増補分（45点）を含む v3 があればそちらを優先する。
# v2 は v1 の 64 点しか無いので，増補後のコーパスで v2 を使うと
# 突合が外れて period も genre も空になる（Step 3 で v3 を作る）。
# Step 3 で自分が作った v3 は *_local.csv に書かれる（配布版は上書きしない。
# 上書きすると git pull のたびに衝突する）。自分の版 > 配布版 v3 > v2 の順。
for _m in ('corpus_metadata_v3_local.csv', 'corpus_metadata_v3.csv',
           'corpus_metadata_v2.csv'):
    META = ROOT / 'metadata' / _m
    if META.exists():
        break

OUT = ROOT / 'results' / ME
OUT.mkdir(parents=True, exist_ok=True)
print('OUT  =', OUT)


## 1. 書誌索引 — 一次資料としての CSV

青空文庫は公開中の全作品（約 2万点）について，
作品ID・作品名・**初出**・**分類番号(NDC)**・文字遣い種別・底本・親本・
テキスト/XHTML の URL を1つの CSV で配布している。

`list_person_all_extended_utf8.zip`

**v1 の `year` 列の誤りは，この索引を使っていれば起きなかった種類の誤りである。**
図書カードを1件ずつ読む必要もない。

In [ ]:
# 索引の取得（最初の一度だけ。約 20 MB）
DATA = ROOT/'data'/'aozora'
run_script('02_fetch_aozora.py', 'index', '--out', DATA)

In [ ]:
idx = pd.read_csv(DATA/'list_person_all_extended_utf8.csv',
                  encoding='utf-8-sig', low_memory=False)
print(idx.shape)
print([c for c in idx.columns][:20])
# 森鴎外の作品を覗いてみる
m = idx[(idx['姓']=='森') & (idx['名']=='鴎外')]
show(m[['作品ID','作品名','初出','分類番号','文字遣い種別']].head(15),
     caption='青空文庫索引から引いた森鴎外の作品（先頭15件）')

### 演習 1 — v1 の誤りを索引で確かめる

小栗虫太郎『潜航艇「鷹の城」』を索引から引き，`初出` 欄と `底本初版発行年1` を
比べよ。v1 が記録していた 1977 はどちらか。

In [ ]:
q = idx[idx['作品名'].astype(str).str.contains('鷹の城', na=False)]
cols = [c for c in idx.columns if c in
        ['作品ID','作品名','初出','分類番号','文字遣い種別',
         '底本名1','底本出版社名1','底本初版発行年1']]
show(q[cols].T.reset_index().rename(columns={'index':'項目'}),
     caption='『潜航艇「鷹の城」』の索引の記述（列を縦に倒した）')

## 2. マニフェストによる取得

取りたい作品は `config/corpus_manifest.tsv` に **作者名＋作品名**で書く。
作品IDが分かっていれば書いてもよい。スクリプトが索引と突き合わせて
ID・URL・書誌を解決する。

In [ ]:
man = pd.read_csv(ROOT/'config'/'corpus_manifest.tsv', sep='\t', comment='#')
show(man['set'].value_counts().rename_axis('set').reset_index(name='件数'),
     caption='マニフェストの内訳')
show(man.head(8), caption='マニフェストの先頭8行')

In [ ]:
# まず解決だけ試す（ダウンロードしない）
run_script('02_fetch_aozora.py', 'resolve',
           '--manifest', ROOT/'config'/'corpus_manifest.tsv', '--out', DATA)

In [ ]:
# 未解決の一覧を表で確認する（resolve を実行すると作られる）
p = DATA/'unresolved.csv'
if need(p):
    un = pd.read_csv(p)
    show(un.cause.value_counts().rename_axis('原因').reset_index(name='件数'),
         caption='未解決の内訳')
    # 原因ごとにまとめ，原因名はブロックの先頭行だけに出す。
    # 候補は長いので 160 字で切る（全文は unresolved.csv にある）。
    rows = []
    for cause in ['title_mismatch', 'author_differs', 'author_not_found']:
        d = un[un.cause == cause]
        for i, (_, r) in enumerate(d.iterrows()):
            cand = r.candidates if isinstance(r.candidates, str) else ''
            rows.append({'原因': cause if i == 0 else '', 'set': r['set'],
                         '作家': r.author_ja, '作品': r.title,
                         '候補': cand[:160]})
    if rows:
        show(pd.DataFrame(rows), caption='未解決の一覧（原因ごと）')
        print('**「候補」が出ているものは綴りの違いである。** '
              'マニフェストの表記を索引に合わせるか，作品IDを直接書く。')
else:
    print('unresolved.csv がまだありません。上の resolve セルを実行してください。')

**未解決行は原因つきで表示される。** 原因は3つに分かれ，対処が違う。

| 原因 | 意味 | 対処 |
|---|---|---|
| `title_mismatch` | 著者は登録あり，作品名が一致しない | 候補を見てマニフェストの `title` を直す |
| `author_differs` | 同名作品が別の著者名の下にある | **翻訳作品**は原著者の下に入る（『即興詩人』→アンデルセン，『小公子』→バーネット）。`author_ja` を直し，訳者は `note` へ |
| `author_not_found` | 著者そのものが青空文庫にない | **著作権保護期間中（没後70年）**。作品名を直しても解決しない。マニフェストから外す |

`title_mismatch` の典型は，作品名の表記ゆれ（『不如帰』vs『小説 不如帰』），
副題つき（『半七捕物帳 69 白蝶怪』），分冊（『夜明け前 01 第一部上』），
新字版・旧字版の併存（啄木『鳥影』）である。

一覧は `data/aozora/unresolved.csv` にも落ちる。
**core の行が混じっていたら最優先で直す**こと。prio1〜4 は増補候補なので，
当面は次のセルの `--set core` で現行64点だけ先に取得してよい。

**この突き合わせ作業そのものがコーパス構築の実務である。**

In [ ]:
# 本文（XHTML）の取得。1秒/件の間隔を空けるので，100件で約2分かかる。
# 共用 iMac では代表者1名が実行し，data/aozora/xhtml を共有すること。
RUN_FETCH = False        # ← 実行するときだけ True に
ONLY_CORE = True         # True なら現行64点（set=core）だけ取得する

if RUN_FETCH:
    opts = ['--set', 'core'] if ONLY_CORE else []
    run_script('02_fetch_aozora.py', 'works',
               '--manifest', ROOT/'config'/'corpus_manifest.tsv',
               '--out', DATA, *opts)
else:
    print('RUN_FETCH = False のままです。取得するときは True にしてください。')

## 3. 外字の復元 — 面区点から Unicode へ

青空文庫の外字注記は JIS X 0213 の **面-区-点** を与えている。

```
※［＃「广＋(炎/鳥)」、第4水準2-81-40］
                      └面┘└区┘└点┘
```

Python の `euc_jis_2004` コーデックを使うと，面区点はバイト列に直して
そのまま復号できる。

| 面 | バイト列 |
|---|---|
| 面1 | `0xA0+区, 0xA0+点` |
| 面2 | `0x8F, 0xA0+区, 0xA0+点` |

`scripts/lib/aozora.py` の `menkuten_to_char()` がこれを実装している。

### 注記は3つの形式がある

面区点を与えてくれるのは**いちばん親切な形式**にすぎない。実際には3つある。

| 形式 | 例 | 解決法 |
|---|---|---|
| 面区点 | `※［＃「广＋(炎/鳥)」、第4水準2-81-40］` | `euc_jis_2004` で復号 |
| Unicode | `※［＃「女＋（而／大）」、U+5A86、7巻-16-下-14］` | `chr(0x5A86)` |
| **字の名前だけ** | `※［＃小書き片仮名ヲ、160-9］` | 対応表を引く。無ければ欠字 |

3つめが曲者である。JIS X 0213 に無い字なので青空文庫は番号を書けない。
末尾の `160-9` は**底本のページ-行**であって面区点ではない。これを面区点と
読み違えたり，注記の中身に「水準」という語が無いことを理由に外字扱いを
しなかったりすると，`※` が本文に残る。`※` は UniDic では記号1字として
数えられるので，**文字が失われたことが誰にも気づかれない**。

さらに，`※` が**ルビの基底文字の中**に入ることもある。

```html
<ruby rt="どんさうじゆけい">※相寿桂</ruby><span class="notes">
［＃「女＋（而／大）」、U+5A86、7巻-16-下-14］</span>
```

注記の直前が `</ruby>` なので「直前の文字が `※` か」だけを見ていると
取り逃がす。森鴎外『伊沢蘭軒』だけで 74 箇所がこの形である。

In [ ]:
from lib.aozora import menkuten_to_char, resolve_gaiji

tests = [(1,15,25),(2,78,35),(2,81,40),(1,84,7),(2,3,72),(1,14,22)]
show(pd.DataFrame([{'面': m, '区': k, '点': t,
                    '復元した字': menkuten_to_char(m,k,t)}
                   for m,k,t in tests]),
     caption='面区点 → 文字（JIS X 0213 の面区点番号を Unicode に直す）')

descs = ['「广＋(炎/鳥)」、第4水準2-81-40',            # 面区点
         '「女＋（而／大）」、U+5A86、7巻-16-下-14',    # Unicode
         '「にんべん＋弖」、第3水準1-14-22',
         '小書き片仮名ヲ、160-9',                      # 字名のみ（解決できる）
         '「弋＋頁」、239-6',                          # 字名のみ（解決できない）
         ]
rows = []
for desc in descs:
    ch, kind = resolve_gaiji(desc)
    rows.append({'入力者注の記述': desc, '復元': ch if ch else '（不能）',
                 '手がかり': kind})
show(pd.DataFrame(rows), caption='注の記述からの復元（どの手がかりで解けたか）')
print('「手がかり」の列が復元の根拠である。**同じ字が別の記述で書かれること'
      'があるので，根拠を残さないと再現できない。**')

### 演習 2 — 失われた文字を数え，どこで失われたかを言う

`03_aozora2xml.py` の変換レポートには `gaiji_resolved` と
`gaiji_unresolved` が出る。両者の意味は**まったく違う**。

- `gaiji_resolved` … 実体を復元して `<g ref="…">侔</g>` に入れた
- `gaiji_unresolved` … 実体が分からないので `<g ref="unresolved" n="注記">`
  として残した。正規化後は `〓` になる

そして，正規化後のテクストに残る `※` は**第3のもの**である。

1. 正規化後テクスト（`data/plain/full/`）の `※` と `〓` をそれぞれ数えよ。
2. `※` が残っている作品を1つ選び，青空文庫の原文を見て，それが
   **(a) 外字マーカの消し残し** なのか **(b) 底本そのものが使う記号**
   （編者注の印，伏字）なのかを判定せよ。
3. (b) だった場合，それは欠陥か。なぜそう言えるか。

**ヒント**: 海野十三『敗戦日記』には `（※マリアナ基地からの…）` という
補注が 100 箇所あまりある。これは底本の編者（橋本哲男）が付けたもので，
**本文の一部**である。外字マーカの消し残しではない。
なお，このファイルは `<div class="main_text">` を持たない旧形式なので，
`03_aozora2xml.py` は `<hr>` を手掛かりに奥付を落としている。奥付にも外字注記の `※` があるため，**奥付を落とさないと (a) と (b) が
混ざって数えられない**。

## 4. XML への変換

`scripts/03_aozora2xml.py` が XHTML を TEI 風 XML に変換する。

```xml
<p>池田氏、名は<g ref="1-14-22">侔</g>、字は河澄。</p>
<p><said>「わたくしは<ruby rt="こ">此</ruby>に記述したい。」</said>と云つた。</p>
<p><quote>『武鑑』</quote>に接続する<note type="textual">「○」は底本では「●」</note>。</p>
<p><quote type="embedded" part="I">「……私はこの夏あなたから手紙を受け取りました。</p>
<p><quote type="embedded" part="M">妻は<said>「もう寝ましょう」</said>と云った。</p>
```

要点は5つ。

- `<g>` に外字の**実体**が入る（`ref` に面区点を残すので検証できる）
- `<ruby rt="...">` で読みを保持したまま基底文字を本文に残す
- `<said>` で**会話と地の文を分離**する。語りの人称を測るとき必須になる
- `@part="I|M|F"` は段落をまたぐ会話の断片。XML の入れ子を段落構造と
  交差させられないので，段落末で閉じ，次の段落で開き直す
- `<quote type="embedded">` は8段落以上にわたる引用，つまり書簡・演説・
  手記といった**埋め込まれたテクスト**。その内側の対話は `<said>` になる

最後の2つは，底本の組版慣習に合わせるために必要になった。
`「` で始まり `」` で閉じる，という素朴な前提は日本語の底本では成り立たない。

In [ ]:
XHTML = DATA/'xhtml'
XML   = ROOT/'data'/'xml'
if need(XHTML, '上の RUN_FETCH を True にして青空文庫から取得すること'):
    run_script('03_aozora2xml.py', '--in', XHTML,
               '--log', DATA/'fetch_log.csv', '--out', XML)
else:
    print('XHTML がまだありません。上の RUN_FETCH を True にして取得してください。')

### 分冊を1作品にまとめる

青空文庫は長篇を分冊ごとに別カードで公開する。『夜明け前』は4カード，
『家』は上下2カードである。**そのまま進めると，分析の単位が「作品」では
なく「冊」になる。**

- Delta や doc2vec の最近傍が，同じ作品の別の巻になる。
  Step 4 の「最近傍が同じ作家である割合」は自明に上がり，指標として死ぬ
- 1作家あたりの作品数が水増しされる（島崎藤村は分冊のせいだけで9点あった）
- 語数・TTR・文体指標が巻ごとにばらけ，他の作品と比べられない

TEI 的にも，分冊は1つの `<text>` の中の `<div type="volume">` であって
別の文書ではない。**XML の段階でまとめる**のが正しい。ここでまとめておけば，
04・05・06・99 はすべて「1作品1ファイル」の前提のまま動く。

まとめる組合せは `config/merge_volumes.tsv` に書いてある。もとの巻別 XML は
`data/xml/_volumes/` に退避されるので，巻ごとの比較をしたくなれば戻せる。

In [ ]:
if need(XML, 'Step 2 の 03_aozora2xml.py を先に走らせること'):
    # まず確認だけ（--dry-run）。何が何にまとまるかを読んでから本番を走らせる
    run_script('03b_merge_volumes.py', '--xml', XML,
               '--config', ROOT/'config'/'merge_volumes.tsv', '--dry-run')

In [ ]:
if need(XML, 'Step 2 の 03_aozora2xml.py を先に走らせること'):
    run_script('03b_merge_volumes.py', '--xml', XML,
               '--config', ROOT/'config'/'merge_volumes.tsv')

In [ ]:
# 変換レポート：外字がどれだけ復元できたか
p = XML/'conversion_report.csv'
if need(p, 'Step 2 の 03_aozora2xml.py を先に走らせること'):
    rep = pd.read_csv(p)
    print('外字 復元合計 =', rep.gaiji_resolved.sum(),
          '／ 未解決 =', rep.gaiji_unresolved.sum())
    show(rep.nlargest(12,'gaiji_resolved')[
            ['author','title','gaiji_resolved','gaiji_unresolved','ruby','said']]
         .rename(columns={'author':'作家','title':'作品',
                          'gaiji_resolved':'外字 復元','gaiji_unresolved':'未解決',
                          'ruby':'ルビ','said':'会話'}),
         caption='外字の多い作品（復元数の上位12件）')

### 会話標示の診断を読む

変換レポートには会話の標示に関する列が並ぶ。素朴な実装では出ない列である。

| 列 | 意味 |
|---|---|
| `speech_mark` | その作品の**第一階層の会話符**。`「」` か `『』` |
| `said` | 会話の数（段落をまたぐものも1と数える） |
| `said_parts` | `<said>` 要素の数。断片化したぶん `said` より多い |
| `said_density` | 1万字あたりの会話数 |
| `said_cont` | 継続引用符で接続した回数 |
| `quote_embedded` | 埋め込みテクストの数 |
| `said_unclosed` | 閉じ括弧のない開き括弧の数 |
| `speech_markup` | 標示が信用できるか。`full` / `partial` / `none` |

`speech_mark` が `『』` になる作品があることに注意せよ。島崎藤村『破戒』は
`「」` が 47 に対し `『』` が 1,574 で，会話のほぼ全部が `『』` である。
一律に `「」` を会話と見ると，この作品の会話が**丸ごと 0 件**になる。

In [ ]:
if need(p):
    cols = ['author','title','speech_mark','speech_markup','said',
            'said_parts','said_density','said_cont','quote_embedded',
            'said_unclosed']
    cols = [c for c in cols if c in rep.columns]
    show(rep[rep.speech_mark == '『』'][cols],
         caption='第一階層の会話符が『』の作品（「」で数えると会話が 0 件になる）')
    show(rep.nlargest(8, 'said_cont')[cols],
         caption='継続引用符・埋め込みテクストが多い作品')
    show(rep.nlargest(6, 'said_unclosed')[cols],
         caption='閉じ括弧のない開き括弧が多い作品（標示の信頼性が落ちる）')

### 演習 3 — 会話文比率を出す。そして「0」と「測れない」を区別する

`<said>` があるので，作品ごとの会話文比率が計算できる。これは
**ジャンルを最もよく分ける文体指標の一つ**である（戯曲＞小説＞随筆＞論説）。

ただし注意すべきことがある。**比率が 0 であることと，比率が測れないことは違う。**
底本が会話符を使わない作品がある。樋口一葉『たけくらべ』『にごりえ』は
会話に括弧を用いず地の文に溶け込ませる。岡本綺堂『修禅寺物語』は戯曲で，
会話は話者名と字下げで示される。福沢諭吉『福翁自伝』は口述筆記で，
`「` で話を起こして「と云う」で閉じ `」` を置かない（本文中 `「` 330 に対し
`」` 38）。これらを「会話文比率 0」として分析に入れると，
**文体指標も話法の通時変化も系統的に歪む**。

`03_aozora2xml.py` は `teiHeader` の `<catRef scheme="speech_markup">` に
`full` / `partial` / `none` を記録している。`full` 以外は欠測として扱うこと。

In [ ]:
import xml.etree.ElementTree as ET

def speech_markup(root):
    for cr in root.iter('catRef'):
        if cr.get('scheme') == 'speech_markup':
            return cr.get('target')
    return 'full'

rows = []
for f in sorted(XML.glob('*.xml')):
    root = ET.parse(f).getroot()
    body = root.find('.//body')
    all_text  = ''.join(body.itertext())
    said_text = ''.join(''.join(s.itertext()) for s in body.iter('said'))
    grade = speech_markup(root)
    rows.append({'file': f.name,
                 # 作者も出す。『武蔵野』は山田美妙と国木田独歩の2点あり，
                 # 作品名だけでは区別できない
                 'author': root.findtext('.//author',''),
                 'title': root.findtext('.//title',''),
                 'chars': len(all_text),
                 'markup': grade,
                 # full 以外は NaN にする。0 で埋めてはいけない
                 'speech_ratio': (len(said_text)/max(1,len(all_text))
                                  if grade == 'full' else float('nan'))})
if rows:
    df = pd.DataFrame(rows)
    show(df.markup.value_counts().rename_axis('標示の程度')
           .reset_index(name='作品数'), caption='会話標示の内訳')
    show(df[df.markup != 'full'][['author','title','markup','chars']]
         .rename(columns={'author':'作家','title':'作品',
                          'markup':'標示の程度','chars':'字数'}),
         caption='会話文比率を**欠測**にした作品（0 で埋めてはいけない）',
         fmt={'字数':'{:,.0f}'})
    d = df.dropna(subset=['speech_ratio']).sort_values('speech_ratio', ascending=False)
    ends = pd.concat([d.head(8), d.tail(8)])[
        ['author','title','markup','speech_ratio']].rename(
        columns={'author':'作家','title':'作品','markup':'標示の程度',
                 'speech_ratio':'会話文比率'})
    show(ends, caption='会話文比率の上位8件と下位8件',
         fmt={'会話文比率':'{:.1%}'})

## 5. このステップの課題

1. `config/corpus_manifest.tsv` の未解決行を最低3つ解決し，直した行を報告せよ。
2. v1 で `※` になっていた箇所を **3つ**選び，青空文庫の原文から復元した文字と，
   その語の意味を示せ（辞書を引くこと）。3つのうち少なくとも1つは
   **面区点を持たない注記**（`※［＃小書き片仮名ヲ、160-9］` の形）から選ぶこと。
3. XML から会話文比率を計算し，ジャンル別に比較した図を1枚作れ。
   ただし `speech_markup` が `full` でない作品は**図から除き，その旨を図の
   キャプションに書く**こと。0 で埋めて描いた図は誤りである。
4. `scripts/find_unclosed_quotes.py` を `data/xml` にかけ，残った
   `unmarked_paragraph` を1つ選んで青空文庫の原文を確認し，
   **底本の組版のどの慣習に由来するか**を説明せよ。

### このステップの到達点（次へ進む条件）

- `data/xml/` に全件の XML があり，`conversion_report.csv` が出ている
- `gaiji_unresolved` の合計と，その内訳（どの作品のどの字か）を言える
- `speech_markup` が `full` でない作品を列挙でき，理由を説明できる
- `docs/encoding_guidelines.md` を読み終えている
